# Unusual now, or unusual ever?

MichAl Academy, lesson 2.15.

Run each cell with **Shift+Enter**.

Every dataset in this track has been a pile of rows in no particular order.
Almost nothing you will monitor is like that. Logs, traffic, logins and
transactions all arrive in time order, and time changes three things: how you
have to split the data, what "unusual" means, and whether a model trained last
quarter is still describing this one.

**About the data.** No dataset bundled with scikit-learn is a time series, so
this uses the bike sharing demand data from OpenML, which is what
scikit-learn's own documentation uses for exactly this topic. 17,379 hourly
records with a real daily cycle, a real weekly cycle and a real year-on-year
trend. Hourly hire counts are not security data and the arithmetic does not
care.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit

SEED = 0
raw = fetch_openml("Bike_Sharing_Demand", version=2, as_frame=True, parser="auto")
df = raw.frame.copy()
y = df["count"].to_numpy(dtype=float)
X = df.drop(columns=["count"])
print(f"{len(df)} hourly records, in time order")
print(df.head(3).to_string())

cat = [c for c in X.columns if str(X[c].dtype) in ("category", "object")]
num = [c for c in X.columns if c not in cat]
model = Pipeline([
    ("pre", ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat),
        ("num", "passthrough", num),
    ])),
    ("reg", HistGradientBoostingRegressor(random_state=SEED)),
])


## 1. The split that invents fifteen points of accuracy

Lesson 2.3 warned that a random split can leak. On temporal data it does not
just leak a bit, it hands the model the answer: shuffle the rows and the
training set contains the hour before and the hour after almost every test row.

Measure it. Same model, same data, three ways of cutting it.


In [ ]:
for label, cv in (("random KFold, shuffled", KFold(5, shuffle=True, random_state=SEED)),
                  ("random KFold, unshuffled", KFold(5, shuffle=False)),
                  ("TimeSeriesSplit", TimeSeriesSplit(5))):
    s = cross_val_score(model, X, y, cv=cv, scoring="r2")
    print(f"{label:<26} r2 {s.mean():>7.4f}  +/- {s.std():.4f}   folds {np.round(s, 3)}")


**0.9479 shuffled against 0.7944 time-aware.** Fifteen points of r-squared,
manufactured entirely by the order the rows were in.

Now look at the second column, because it is the worse half. The shuffled
estimate has a standard deviation of **0.0026** and the honest one **0.1286**.
The wrong answer arrived fifty times more precisely than the right one, and
precision is what makes a number persuasive. A shuffled split does not just
flatter your model, it flatters your confidence in the flattery.

The unshuffled KFold in the middle, at 0.8339, is worth understanding: it keeps
each fold contiguous but still trains on later data when scoring earlier folds.
Half a fix.

**`TimeSeriesSplit` only ever trains on the past.** On anything with a
timestamp, that is the default, and the burden of proof is on anyone who wants
to shuffle.


## 2. Most of the signal is the clock

Before asking what is unusual, find out how much of the variation is simply
what time it is.


In [ ]:
hour = df["hour"].to_numpy(dtype=int)
dow = pd.to_numeric(df["weekday"], errors="coerce").to_numpy(dtype=int)
how = dow * 24 + hour                        # hour of the week, 0 to 167

by_hour = np.array([y[hour == h].mean() for h in range(24)])
by_how = np.array([y[how == k].mean() if (how == k).any() else y.mean()
                   for k in range(168)])

print(f"variance of hourly demand                     {y.var():>10.1f}")
print(f"after removing each hour-of-day's mean        {(y - by_hour[hour]).var():>10.1f}")
print(f"after removing each hour-of-week's mean       {(y - by_how[how]).var():>10.1f}")
print(f"\nhour of day alone explains  {1 - (y - by_hour[hour]).var() / y.var():>6.1%}")
print(f"hour of week explains       {1 - (y - by_how[how]).var() / y.var():>6.1%}")


Hour of the week explains **63.3%** of the variance on its own, with no model
and no features beyond a clock and a calendar.

Which sets up the whole problem. If two thirds of the variation is the
timetable, then a detector that does not know the time is mostly detecting the
time.


## 3. Three definitions of unusual, agreeing on almost nothing

Take the 200 most unusual hours in the series by three definitions:

1. **Furthest from the overall mean.** The obvious one.
2. **Furthest from its own hour of the week.** Compare a Tuesday 08:00 against
   other Tuesday 08:00s.
3. **Furthest in standard deviations of its own hour of the week.** Because
   rush hour is not only busier, it is more variable, so a hundred hires above
   normal means something different at 03:00 than at 18:00.


In [ ]:
mu = by_how
sd = np.array([y[how == k].std() if (how == k).sum() > 1 else y.std() for k in range(168)])
sd = np.where(sd < 1e-6, y.std(), sd)

TOP = 200
scores = {
    "distance from the overall mean": np.abs(y - y.mean()),
    "distance from its hour-of-week mean": np.abs(y - mu[how]),
    "that distance in hour-of-week sds": np.abs(y - mu[how]) / sd[how],
}
picks = {}
print(f"{'definition':<38} {'hours of day':>13} {'mean demand':>12}")
for label, sc in scores.items():
    idx = np.argsort(-sc)[:TOP]
    picks[label] = set(idx.tolist())
    covered = int((np.bincount(hour[idx], minlength=24) > 0).sum())
    print(f"{label:<38} {covered:>10} of 24 {y[idx].mean():>12.1f}")
print(f"\nthe series averages {y.mean():.1f} hires an hour")

keys = list(scores)
print()
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        shared = len(picks[keys[i]] & picks[keys[j]])
        print(f"definitions {i + 1} and {j + 1} share {shared} of their {TOP} picks")


Read the overlaps first. **Definitions 1 and 3 share 2 picks out of 200.**

They are both "the two hundred most unusual hours in this series" and they are
almost disjoint sets. So "unusual" is not one thing, and if somebody hands you
an anomaly detector without saying which definition it implements, you do not
know what it will alert on.

Then read the hours-of-day column, which explains why.

**Definition 1 flags 6 of the 24 hours** and the mean demand of what it flags
is 835.8 against a series average of 189.5. It has found rush hour. Every
alert is "the busy period was busy", which is true and worthless.

**Definition 3 flags all 24 hours** and the mean demand of what it flags is
212.9, close to the series average. It is no longer measuring the clock.


In [ ]:
# So what does definition 3 actually find?
z = np.abs(y - mu[how]) / sd[how]
DAYS = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
print(f"{'when':<22} {'hires':>6} {'typical':>9} {'sd':>7} {'off by':>8} {'weather':>8}")
for k in np.argsort(-z)[:8]:
    print(f"{DAYS[dow[k]] + f' {hour[k]:02d}:00':<22} {y[k]:>6.0f} {mu[how[k]]:>9.1f}"
          f" {sd[how[k]]:>7.1f} {z[k]:>6.1f}sd {str(df['weather'].iloc[k]):>8}")


A Wednesday midnight with 283 hires where 35 is normal. A Wednesday 02:00 with
93 where 8 is normal. Something happened that night, and definitions 1 and 2
cannot see it: 283 hires is far below the overall mean, so definition 1 ranks it
as an ordinary hour.

**This is the temporal version of lesson 2.7's base rate argument.** Judging a
value against the wrong population produces a confident, useless answer.


## 4. Does last quarter's model still describe this one?

Train on the first half of the series and score each later block. The usual
expectation is a decay curve.


In [ ]:
cut = len(df) // 2
fitted = model.fit(X.iloc[:cut], y[:cut])
edges = np.linspace(cut, len(df), 9).astype(int)

print(f"trained on hours 0 to {cut}")
print(f"{'block':>16} {'r2':>8} {'actual mean':>12} {'predicted':>11} {'bias':>8}")
for i in range(8):
    a, b = edges[i], edges[i + 1]
    pred = fitted.predict(X.iloc[a:b])
    print(f"{f'{a}-{b}':>16} {fitted.score(X.iloc[a:b], y[a:b]):>8.4f}"
          f" {y[a:b].mean():>12.1f} {pred.mean():>11.1f}"
          f" {pred.mean() - y[a:b].mean():>8.1f}")


There is no decay curve. R-squared goes 0.49, 0.41, 0.65, 0.73, 0.66, 0.65,
0.61, 0.68: worst at the start, then better, then flat. If you had been
watching r-squared you would have concluded the model was fine and improving.

Now read the bias column, which is negative in **every single block**. The
model under-predicts by 51 to 104 hires, consistently, and never once
over-predicts.


In [ ]:
pred_later = fitted.predict(X.iloc[cut:])
print(f"training period mean demand     {y[:cut].mean():>7.1f}")
print(f"later period mean demand        {y[cut:].mean():>7.1f}"
      f"   ({y[cut:].mean() / y[:cut].mean():.2f}x)")
print(f"what the model predicts for it  {pred_later.mean():>7.1f}"
      f"   bias {pred_later.mean() - y[cut:].mean():+.1f}")
print(f"\nhighest demand the model ever predicts {pred_later.max():>6.0f}")
print(f"highest demand in its training data    {y[:cut].max():>6.0f}")
print(f"highest demand in the later period     {y[cut:].max():>6.0f}")


There it is. Demand grew **1.64 times** between the two halves, and the model
under-predicts the later period by 35%.

And the last three lines are the mechanism, which is structural rather than a
matter of tuning. **A tree predicts the average of the training rows in a leaf,
so it can never output a value above the highest it was trained on.** Its
training data topped out at 651, the later period reaches 977, and the model's
largest prediction is 598. Not "struggles to extrapolate". Cannot.

Two consequences worth carrying:

**Watch the bias, not only the score.** R-squared measures how well the
predictions track the ups and downs, and it stayed respectable while every
prediction was too low. Lesson 2.12's habit again: print the thing that goes
wrong quietly.

**For a trending series, model the trend or remove it.** Fit on the difference
from the previous period, or add an explicit time index and use a model that
can extrapolate along it, which trees cannot. Linear regression can, which is
one of the few places <Lesson id="2.4" />'s model is the more powerful choice.


## What to take from this

| Claim | What we measured |
|---|---|
| A shuffled split is fine if the model is good | It reported 0.9479 against 0.7944 for a time-aware split |
| At least the shuffled estimate was consistent | Its sd was 0.0026 against 0.1286. Fifty times more confident and wrong |
| Anomaly detection finds anomalies | Definition 1 flagged rush hour: 6 hours of the day, mean demand 835.8 against 189.5 |
| The definitions of "unusual" broadly agree | The first and third shared 2 of their 200 picks |
| Subtracting the seasonal mean is enough | It got from 6 hours of the day to 11. Dividing by the seasonal sd got to 24 |
| A stale model shows up as a falling score | R-squared went 0.49, 0.41, 0.65, 0.73 while the bias was negative in every block |
| Trees struggle to extrapolate a trend | They cannot. Trained data topped at 651, the model's largest prediction was 598 |


## Try this

1. Add `hour` and `weekday` as explicit features to the seasonal baseline and
   refit with `TimeSeriesSplit`. How much of the 0.7944 was the clock?
2. Replace the boosted tree in section 4 with `LinearRegression` on a numeric
   time index plus the hour dummies. Does the bias go away? This is the
   extrapolation argument, checked rather than accepted.
3. Take definition 3 and set the threshold from a cost, using lesson 2.8's
   arithmetic. At 3 standard deviations, how many alerts an hour does this
   series generate, and would a person keep up?
